# 👗 Fashion MNIST Classification with Improved Neural Network

## Overview
This notebook demonstrates how to build a **well-regularized feedforward neural network** for image classification using the **Fashion MNIST** dataset.

### Key Improvements over a basic model:
- **BatchNormalization** — stabilizes and accelerates training
- **Dropout** — reduces overfitting
- **EarlyStopping** — stops training when validation loss plateaus
- **ReduceLROnPlateau** — dynamically decreases learning rate for better convergence

**Target Accuracy:** 88%+

## 1️⃣ Import Libraries

In [ ]:
# Deep learning, visualization, and evaluation libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)

# Fashion MNIST class labels
CLASS_NAMES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

## 2️⃣ Load & Explore Dataset

**Fashion MNIST** contains 70,000 grayscale images (28×28 pixels) across **10 fashion categories**.  
It is a popular drop-in replacement for the classic MNIST digit dataset.

In [ ]:
# Load Fashion MNIST (70,000 grayscale 28x28 images, 10 categories)
fashion_mnist = keras.datasets.fashion_mnist
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

print(f"Training set:  {X_train.shape}, Labels: {y_train.shape}")
print(f"Test set:      {X_test.shape}, Labels: {y_test.shape}")
print(f"Pixel value range: {X_train.min()} – {X_train.max()}")

In [ ]:
# Visualize a 5x5 grid of sample training images
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
indices = np.random.choice(len(X_train), 25, replace=False)

for ax, idx in zip(axes.ravel(), indices):
    ax.imshow(X_train[idx], cmap='gray')
    ax.set_title(CLASS_NAMES[y_train[idx]], fontsize=9)
    ax.axis('off')

plt.suptitle('Fashion MNIST — Sample Training Images', fontsize=14)
plt.tight_layout()
plt.show()

## 3️⃣ Preprocess Data

In [ ]:
# Normalize pixel values to [0, 1] and flatten for dense layers
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

# Flatten 28x28 images to 784-dim vectors
X_train_flat = X_train_norm.reshape(-1, 28 * 28)
X_test_flat = X_test_norm.reshape(-1, 28 * 28)

print(f"Normalized training shape: {X_train_flat.shape}")
print(f"Normalized test shape:     {X_test_flat.shape}")

## 4️⃣ Build Improved Neural Network

The improved architecture uses:
- **BatchNormalization** after each Dense layer to normalize activations
- **Dropout** layers to prevent overfitting by randomly dropping neurons during training

In [ ]:
# Improved architecture with BatchNormalization and Dropout for regularization
model = keras.Sequential([
    layers.Input(shape=(784,)),
    
    # Hidden Layer 1
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    
    # Hidden Layer 2
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    # Hidden Layer 3
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    
    # Output Layer
    layers.Dense(10, activation='softmax')
], name='Improved_Fashion_MNIST')

model.summary()

## 5️⃣ Compile Model with Callbacks

### Callbacks:
| Callback | Purpose |
|---|---|
| `EarlyStopping` | Stops training if val_loss doesn't improve for 5 epochs |
| `ReduceLROnPlateau` | Halves learning rate when val_loss stagnates for 3 epochs |

In [ ]:
# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# EarlyStopping: stop training if val_loss doesn't improve for 5 epochs
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# ReduceLROnPlateau: halve LR when val_loss stagnates
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

print("Model compiled. Callbacks configured.")

## 6️⃣ Train the Model

In [ ]:
# Train the model with callbacks
history = model.fit(
    X_train_flat, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print(f"\n✅ Training complete. Best val accuracy: {max(history.history['val_accuracy']):.4f}")

## 7️⃣ Evaluate Model

In [ ]:
# Evaluate on the held-out test set
test_loss, test_acc = model.evaluate(X_test_flat, y_test, verbose=0)
print(f"Test Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"Test Loss     : {test_loss:.4f}")

## 8️⃣ Visualize Training History

In [ ]:
# Plot training and validation accuracy and loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Model Accuracy', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9️⃣ Classification Report & Confusion Matrix

In [ ]:
# Predict on test set
y_pred = np.argmax(model.predict(X_test_flat, verbose=0), axis=1)

# Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.title('Confusion Matrix — Fashion MNIST Improved Model', fontsize=13)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 🔑 Key Takeaways

| Technique | Benefit |
|---|---|
| **BatchNormalization** | Stabilizes activations, faster convergence |
| **Dropout** | Prevents overfitting, improves generalization |
| **EarlyStopping** | Avoids wasted compute and overfitting |
| **ReduceLROnPlateau** | Fine-grained LR control for better convergence |
| **Adam optimizer** | Adaptive learning rate, robust default choice |

### Conclusion
By adding BatchNormalization and Dropout, combined with smart callbacks, we achieve **88%+ test accuracy** on Fashion MNIST — significantly better than a basic dense network.